# i-FairSC Real Dataset Experiment Notebook
Run i-FairSC on HighSchool dataset (or any graph data).

In [ ]:
!pip install numpy scipy scikit-learn networkx pandas

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.cluster import KMeans
from scipy.linalg import eigh
import time

## Load Dataset (edit path)

In [ ]:
edges = pd.read_csv('edges.csv')
meta = pd.read_csv('metadata.csv')

## Build Graph

In [ ]:
G = nx.Graph()
for _, row in edges.iterrows():
    G.add_edge(int(row['node_i']), int(row['node_j']))

nodes = list(G.nodes())
node_to_idx = {n:i for i,n in enumerate(nodes)}

n = len(nodes)
W = np.zeros((n,n))

for u,v in G.edges():
    i,j = node_to_idx[u], node_to_idx[v]
    W[i,j] = 1
    W[j,i] = 1

## Load Groups

In [ ]:
groups = np.zeros(n, dtype=int)
for _, row in meta.iterrows():
    if row['node'] in node_to_idx:
        groups[node_to_idx[row['node']]] = int(row['group'])

## Helper Functions

In [ ]:
def compute_ncut(W, labels):
    ncut = 0
    for c in np.unique(labels):
        idx = np.where(labels==c)[0]
        not_idx = np.where(labels!=c)[0]
        cut = np.sum(W[np.ix_(idx,not_idx)])
        vol = np.sum(W[idx])
        ncut += cut/(vol+1e-8)
    return ncut

def compute_balance(labels, groups):
    scores = []
    for c in np.unique(labels):
        idx = np.where(labels==c)[0]
        g = np.bincount(groups[idx])
        scores.append(np.min(g/len(idx)))
    return np.mean(scores)

## i-FairSC

In [ ]:
def ifairsc(W, groups, k=5, alpha=0.5, beta=0.5):
    start = time.time()
    G = nx.from_numpy_array(W)

    clique_mat = np.zeros_like(W)
    for c in nx.enumerate_all_cliques(G):
        if len(c)>=3:
            for i in c:
                for j in c:
                    if i!=j:
                        clique_mat[i,j]+=1

    W_new = W + alpha*clique_mat
    D = np.diag(W_new.sum(axis=1))
    L = D - W_new

    F = np.zeros((len(groups), len(np.unique(groups))))
    for i,g in enumerate(groups):
        F[i,g]=1

    L_total = L + beta*(F@F.T)

    eigvals, eigvecs = eigh(L_total)
    U = eigvecs[:,:k]

    labels = KMeans(n_clusters=k, n_init=10).fit_predict(U)

    ncut = compute_ncut(W_new, labels)
    bal = compute_balance(labels, groups)
    t = time.time()-start
    return labels, ncut, bal, t

## Run Experiment

In [ ]:
labels, ncut, bal, runtime = ifairsc(W, groups, k=5)
print('Ncut:', ncut)
print('Balance:', bal)
print('Time:', runtime)